# 05 — Layouts, Containers & Page Structure

> **📓 Notebook · Module 03 · Beginner**  
> *Master sidebar, columns, tabs, expanders, containers, and the layout hierarchy.*

---

## 🎯 Learning Objectives

By the end of this notebook you will be able to:

1. Use `st.sidebar` to organize controls separate from content.
2. Create multi-column layouts with `st.columns()` including gap, alignment, and border options.
3. Group content with `st.tabs()`, `st.expander()`, `st.container()`, and `st.empty()`.
4. Use `st.popover()` and `@st.dialog` for contextual UI.
5. Apply the layout hierarchy to build a structured Data Science dashboard.

## 📋 Prerequisites

- Completed [Notebook 03 — Streamlit Widgets](03_streamlit_widgets.ipynb)
- Understanding of widget return values and the rerun model
- Python basics (functions, loops, conditionals)

---

## 📚 Concept: How Layout Works

Streamlit apps are structured as a **single page** with two main regions:

```
┌──────────┬──────────────────────────┐
│ Sidebar  │       Main Area          │
│ (fixed)  │  (scrollable)            │
│          │                          │
│ Controls │  Content                 │
│ Filters  │  Charts, Tables, Text    │
│ Settings │  Metrics, Downloads      │
└──────────┴──────────────────────────┘
```

**Layout elements** let you subdivide these regions:
- `st.columns()` — side-by-side horizontal divisions
- `st.tabs()` — switchable content panels
- `st.expander()` — collapsible sections
- `st.container()` — logical grouping (no visual change)
- `st.empty()` — a single-element slot that can be replaced
- `st.popover()` — floating content panel
- `@st.dialog` — modal overlay

## 🧠 Intuition: Layout Is Like Furniture in a Room

Think of your Streamlit app as a room:

```
Room = App Page
├── Desk (Sidebar)       → Where you set up tools and controls
├── Wall Art (Title)     → First thing people see
├── Table (Columns)      → Things placed side by side
├── Drawers (Expanders)  → Stuff hidden until opened
├── Flip Chart (Tabs)    → Different views, one at a time
└── Whiteboard (Empty)   → Content that changes dynamically
```

A well-organized room helps you find things quickly. A well-organized app helps users understand data quickly.

---

## 🔧 Build It: Sidebar

The sidebar is the control panel of your app. It stays fixed on the left while the main area scrolls.

In [ ]:
import streamlit as st
import pandas as pd
import numpy as np

st.set_page_config(page_title="Notebook 05", layout="wide")

st.header("Sidebar Basics")
st.markdown("\n".join([
    "The sidebar is your **control panel**."
    "Users expect filters and settings to live here."
]))

# Sidebar controls
st.sidebar.header("Controls")
category = st.sidebar.selectbox("Category", ["All", "Electronics", "Clothing", "Food", "Books"])
price_range = st.sidebar.slider("Price Range", 0, 1000, (50, 500))
show_raw = st.sidebar.checkbox("Show raw data")

# Main area responds to sidebar
st.write(f"**Filters:** {category}, ${price_range[0]}–${price_range[1]}")
if show_raw:
    st.write("Raw data panel would appear here.")

### Sidebar with Form (Batch Filters)

When you have many filters, wrap them in a form so changes are batched into a single rerun.

In [ ]:
with st.sidebar.form("batch_filters"):
    st.subheader("Advanced Filters")
    region = st.selectbox("Region", ["North", "South", "East", "West"])
    min_rating = st.slider("Min Rating", 1.0, 5.0, 3.0, 0.1)
    date_range = st.date_input("Date range")
    apply = st.form_submit_button("Apply Filters")

if apply:
    st.success(f"Filters applied: Region={region}, Rating≥{min_rating}")

---

## 🔧 Build It: Columns

Columns place elements **side by side** — the bread and butter of dashboards.

In [ ]:
st.header("Column Basics")

# Equal columns
col1, col2, col3 = st.columns(3)

with col1:
    st.metric("Revenue", "$45,231", "+5.2%")
with col2:
    st.metric("Users", "1,204", "+12.1%")
with col3:
    st.metric("Conversion", "3.2%", "-0.4%")

st.divider()

# Unequal columns
st.subheader("Unequal Columns (70/30 Split)")
main_area, side_area = st.columns([0.7, 0.3])

with main_area:
    st.write("**Main content:** Charts, visualizations, primary data.")
    np.random.seed(42)
    chart_data = pd.DataFrame(
        np.random.randn(20, 3),
        columns=["Series A", "Series B", "Series C"]
    )
    st.line_chart(chart_data)

with side_area:
    st.write("**Side panel:** Supporting info.")
    st.info("Chart details")
    st.write("- 20 data points")
    st.write("- 3 series")
    st.write("- Normal distribution")

### Column Gap and Alignment

In [ ]:
st.subheader("Column Gaps")

c1, c2, c3 = st.columns(3, gap="large")
c1.metric("Wide Gap", "A")
c2.metric("Between", "B")
c3.metric("Columns", "C")

st.subheader("Vertical Alignment")

left, center, right = st.columns(3, vertical_alignment="center")
left.button("Short button")
center.markdown("This is a much longer piece of text that spans multiple lines")
right.checkbox("Check me")

### Column Borders

Use `border=True` to visually separate columns.

In [ ]:
st.subheader("Bordered Columns")

b1, b2, b3 = st.columns(3, border=True)
with b1:
    st.markdown("**Analysis**")
    st.write("Revenue analysis")
    st.write("- Growth: +8%")
with b2:
    st.markdown("**Forecast**")
    st.write("Q4 projection")
    st.write("- Expected: $1.5M")
with b3:
    st.markdown("**Alerts**")
    st.write("Action items")
    st.write("- 2 items pending")

---

## 🔧 Build It: Tabs

Tabs let users switch between **views of the same data** without scrolling.

In [ ]:
st.header("Tabs")

# Generate sample data
np.random.seed(42)
df = pd.DataFrame({
    "Date": pd.date_range("2026-01-01", periods=12, freq="M"),
    "Revenue": np.random.randint(80000, 150000, 12),
    "Users": np.random.randint(1000, 5000, 12),
    "Orders": np.random.randint(200, 800, 12)
})

tab_chart, tab_table, tab_stats = st.tabs(["📊 Chart", "📋 Table", "📈 Statistics"])

with tab_chart:
    st.subheader("Revenue Over Time")
    st.line_chart(df.set_index("Date")["Revenue"])

with tab_table:
    st.subheader("Monthly Data")
    st.dataframe(df, use_container_width=True, hide_index=True)

with tab_stats:
    st.subheader("Summary Statistics")
    st.write(df.describe())

### Lazy Tab Execution

By default, **all tabs execute on every rerun**. Use `tab.open` to run expensive code only when the tab is selected.

In [ ]:
st.subheader("Lazy Tab Execution")

tab_fast, tab_slow = st.tabs(["Quick View", "Deep Analysis"])

with tab_fast:
    st.write("This always runs — lightweight content.")
    st.write(f"Data shape: {df.shape}")

with tab_slow:
    if tab_slow.open:
        st.write("Expensive analysis only runs when tab is open:")
        # Simulate expensive computation
        correlation = df[["Revenue", "Users", "Orders"]].corr()
        st.dataframe(correlation.style.background_gradient(cmap="coolwarm"))
    else:
        st.write("Switch to this tab to see deep analysis.")

---

## 🔧 Build It: Expander

Expanders let you **hide details** behind a clickable header.

In [ ]:
st.header("Expanders for Progressive Disclosure")

# Summary first
total_revenue = df["Revenue"].sum()
avg_users = df["Users"].mean()
st.metric("Total Revenue (12 months)", f"${total_revenue:,.0f}")
st.metric("Average Monthly Users", f"{avg_users:,.0f}")

# Details on demand
with st.expander("📊 See monthly breakdown"):
    st.dataframe(df, use_container_width=True, hide_index=True)

with st.expander("📈 Year-over-year comparison"):
    st.write("This would contain YoY comparison charts.")
    st.bar_chart(df.set_index("Date")["Revenue"])

with st.expander("📥 Export options"):
    csv = df.to_csv(index=False)
    st.download_button("Download CSV", csv, "data.csv")

---

## 🔧 Build It: Container & Empty

Containers group elements. Empty holds a single replaceable element.

In [ ]:
st.header("Container & Empty")

# Container: write out of order
output = st.container()

# These write in order within the container
output.write("1️⃣ This appears first")
output.write("2️⃣ This appears second")

# But we called them in this order in the script:
# output was defined before the header above

# Empty: replaceable placeholder
placeholder = st.empty()
placeholder.info("Loading data...")

# After "loading", replace the content
placeholder.metric("Result", "42", "+7")

---

## 🔧 Build It: Popover

Popovers are floating panels that open on click — great for settings.

In [ ]:
st.header("Popover for Settings")

with st.popover("⚙️ Display Settings", icon="⚙️"):
    chart_type = st.selectbox("Chart type", ["Line", "Bar", "Area"], key="pop_chart")
    show_grid = st.checkbox("Show gridlines", key="pop_grid")
    colors = st.multiselect("Colors", ["Blue", "Green", "Red"], default=["Blue"], key="pop_colors")

st.write(f"Chart: **{chart_type}**, Grid: **{show_grid}**, Colors: **{colors}**")

# Apply settings
chart_data = pd.DataFrame({
    "A": np.random.randn(10).cumsum(),
    "B": np.random.randn(10).cumsum()
})
if chart_type == "Line":
    st.line_chart(chart_data)
elif chart_type == "Bar":
    st.bar_chart(chart_data)
else:
    st.area_chart(chart_data)

---

## 🔧 Build It: Dialog (Modal)

Dialogs create **modal windows** for focused workflows.

In [ ]:
st.header("Dialog Demo")

@st.dialog("Add Comment")
def add_comment():
    comment = st.text_area("Your comment")
    priority = st.selectbox("Priority", ["Low", "Medium", "High"])
    if st.button("Submit"):
        st.session_state["comments"] = st.session_state.get("comments", [])
        st.session_state["comments"].append({"text": comment, "priority": priority})
        st.rerun()

if st.button("💬 Add Comment"):
    add_comment()

comments = st.session_state.get("comments", [])
if comments:
    for i, c in enumerate(comments, 1):
        st.write(f"**{i}. [{c['priority']}]** {c['text']}")

---

## 🧪 Experiment: Layout Hierarchy in Practice

Let's combine everything into a mini dashboard layout.

In [ ]:
st.header("🧪 Mini Dashboard Layout")

# Sidebar: controls
np.random.seed(42)
with st.sidebar:
    st.header("Dashboard Controls")
    time_period = st.selectbox("Time Period", ["Last 7 days", "Last 30 days", "Last 90 days"])
    metric_choice = st.radio("Primary metric", ["Revenue", "Users", "Orders"], horizontal=True)
    show_forecast = st.checkbox("Show forecast")

# KPI row
k1, k2, k3, k4 = st.columns(4)
k1.metric("Revenue", "$1.2M", "+8.3%", icon="💰")
k2.metric("Users", "45,231", "+1,205", icon="👥")
k3.metric("Orders", "3,847", "+312", icon="📦")
k4.metric("Avg Order", "$312", "+$18", icon="🎯")

# Main content: tabs
tab1, tab2, tab3 = st.tabs(["📈 Trends", "📊 Breakdown", "📋 Details"])

dates = pd.date_range("2026-01-01", periods=30, freq="D")
trend_data = pd.DataFrame({
    "Revenue": np.random.randint(30000, 60000, 30).cumsum(),
    "Users": np.random.randint(1000, 2000, 30).cumsum(),
    "Orders": np.random.randint(80, 200, 30).cumsum()
}, index=dates)

with tab1:
    st.line_chart(trend_data[[metric_choice]])
    if show_forecast:
        st.info("📈 Forecast overlay would appear here")

with tab2:
    c1, c2 = st.columns(2)
    with c1:
        st.bar_chart(trend_data["Revenue"].tail(10))
    with c2:
        st.bar_chart(trend_data["Users"].tail(10))

with tab3:
    st.dataframe(trend_data, use_container_width=True)
    with st.expander("Export"):
        st.download_button("Download CSV", trend_data.to_csv(), "trend.csv")

---

## ⚠️ Common Mistakes

### Mistake 1: Deep Column Nesting

```python
# ❌ Hard to read and maintain
c1, c2 = st.columns(2)
with c1:
    a, b = st.columns(2)
    with a:
        x, y = st.columns(2)

# ✅ Use tabs, expanders, or containers instead
tab1, tab2 = st.tabs(["View A", "View B"])
```

### Mistake 2: Too Many Columns

```python
# ❌ 8 columns — unusable on mobile
c1, c2, c3, c4, c5, c6, c7, c8 = st.columns(8)

# ✅ 4 columns max for KPIs
c1, c2, c3, c4 = st.columns(4)
```

### Mistake 3: Ignoring Alignment

```python
# ❌ Ragged columns look unprofessional
left, right = st.columns(2)
left.button("Click")      # Short
right.text_area("Notes", height=200)  # Tall

# ✅ Aligned columns
left, right = st.columns(2, vertical_alignment="bottom")
```

---

## 🔍 Debugging Tips

| Symptom | Likely Cause | Fix |
|---|---|---|
| Sidebar content appears in main area | Used `st.sidebar.` incorrectly | Ensure `st.sidebar.header()` not `st.header()` |
| Columns don't align vertically | Different content heights | Add `vertical_alignment="center"` |
| All tabs execute on every rerun | Default behavior | Use `if tab.open:` guard |
| Expander content hard to find | Too many expanders | Limit to 3-4 per page |
| Layout breaks on mobile | Too many columns | Use 2-3 columns max |
| Empty() shows stale content | Not replacing correctly | Call `placeholder.empty()` before new content |

---

## ✅ Best Practices

1. **Sidebar for controls, main for content** — always.
2. **Max 4 columns** for KPIs; 2-3 for content splits.
3. **Never nest columns more than one level deep.**
4. **Use `border=True`** on columns when visual separation helps.
5. **Tabs for switching views**, expanders for hiding details.
6. **`gap` parameter** controls space between columns (default `"small"`).
7. **`vertical_alignment`** fixes ragged column heights.
8. **Use `st.empty()`** for dynamic content that replaces in place.
9. **Use `st.popover()`** for floating settings panels.
10. **Use `@st.dialog`** for modal workflows (forms, confirmations).

---

## ✏️ Exercises

### Exercise 1: Layout Factory

Build an app with this exact structure:

```
Sidebar: 3 controls (selectbox, slider, checkbox)
Main: 4 metric cards in a row
Below: 2-column layout — chart on left (70%), table on right (30%)
Bottom: Expandable data table
```

### Exercise 2: Tab Explorer

Create an app with 3 tabs:
- Tab 1: Bar chart
- Tab 2: Line chart
- Tab 3: Data table

Each tab should use the same underlying dataset.

### Exercise 3: Dynamic Dashboard

Build a dashboard where:
1. Sidebar has a `st.form()` with 3 filters
2. KPI row shows 3 metrics that change based on filters
3. Main area has a tab with a chart that updates
4. An expander shows the raw data
5. A popover provides display settings

## 🚀 Challenge Problem

Build a **Product Analytics Dashboard** that uses:
- Sidebar with `st.form()` for batch filtering
- 4 metric cards (Revenue, Users, Conversion, Avg Order)
- 3 tabs (Overview, Trends, Breakdown)
- A 70/30 column split in the Overview tab
- An expander for raw data
- A popover for chart settings
- A dialog for adding notes

Generate synthetic data with `np.random`. Use at least 6 different layout elements.

---

## 📌 Key Takeaways

1. **Sidebar** = control panel; **Main area** = content.
2. **Columns** = side-by-side layout; use `gap`, `vertical_alignment`, `border`.
3. **Tabs** = switchable views; use `tab.open` for lazy execution.
4. **Expanders** = progressive disclosure; show summaries, hide details.
5. **Containers** = logical grouping; **Empty** = replaceable placeholder.
6. **Popover** = floating settings; **Dialog** = modal workflow.
7. **Layout hierarchy:** Page → Sidebar/Main → Tabs → Columns → Content.
8. **Max 4 columns** for dashboards; never nest columns deeply.

---

## 📚 Further Reading

- [Streamlit Layouts API](https://docs.streamlit.io/develop/api-reference/layout)
- [st.columns Reference](https://docs.streamlit.io/develop/api-reference/layout/st.columns)
- [st.tabs Reference](https://docs.streamlit.io/develop/api-reference/layout/st.tabs)
- [st.sidebar Reference](https://docs.streamlit.io/develop/api-reference/layout/st.sidebar)
- [st.expander Reference](https://docs.streamlit.io/develop/api-reference/layout/st.expander)
- [st.popover Reference](https://docs.streamlit.io/develop/api-reference/layout/st.popover)

---

## 🔗 Related Materials

- 📖 Reading: [05 — Layouts, Containers & Page Structure](../readings/05_layouts_and_containers.md)
- 📖 Reading: [06 — Dashboard Design & UI/UX](../readings/06_dashboard_design_ui_ux.md)
- 📓 Notebook: [06 — Data Science Dashboards](06_data_science_dashboards.ipynb)
- ✏️ Exercise: [05 — Layout Basics](../exercises/05_layout_basics.py)
- ✏️ Exercise: [06 — Dashboard Builder](../exercises/06_dashboard_builder.py)
- 🖥️ Demo: [05 — Layouts Demo](../apps/05_layouts_demo.py)
- 🖥️ Demo: [06 — Dashboard Demo](../apps/06_dashboard_demo.py)
- 📝 Quiz: [03 — Layouts & UI/UX](../quizzes/03_layouts_uiux.md)